<a href="https://colab.research.google.com/github/lusiyi0077/Health_LLM_Post-training/blob/main/health_LLM_post_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
training_data = [
    {
        "question": """
A 68-year-old patient with COPD presents with fever,
productive cough, hypoxia, and a new pulmonary opacity.
What should be considered?
""",
        "answer": """
The presentation raises concern for pneumonia, particularly
given the fever, productive cough, hypoxia, and new pulmonary
opacity.

A COPD exacerbation should also remain in the differential
diagnosis.

The available information should be interpreted together with
the patient's full clinical history, examination, imaging, and
laboratory findings before reaching a definitive diagnosis.
"""
    },

    {
        "question": """
A patient presents with chest discomfort and shortness of breath.
What should be considered?
""",
        "answer": """
Several conditions could cause these symptoms, including
cardiovascular and pulmonary causes.

Potentially serious causes should be considered and evaluated
promptly.

Additional information such as symptom characteristics, vital
signs, medical history, physical examination, ECG, and relevant
laboratory testing would be needed to narrow the differential
diagnosis.
"""
    },

    {
        "question": """
A patient with diabetes reports dizziness and confusion.
What additional information would be important?
""",
        "answer": """
Important additional information would include blood glucose,
vital signs, medication use, recent food intake, and the timing
and progression of symptoms.

Both metabolic and non-metabolic causes may need to be
considered.

The available information is insufficient to determine a
specific diagnosis without further clinical evaluation.
"""
    }
]

len(training_data)

3

In [3]:
!pip -q install datasets

In [4]:
from datasets import Dataset

dataset = Dataset.from_list(training_data)

print(dataset)

Dataset({
    features: ['question', 'answer'],
    num_rows: 3
})


In [5]:
print(dataset[0])

{'question': '\nA 68-year-old patient with COPD presents with fever,\nproductive cough, hypoxia, and a new pulmonary opacity.\nWhat should be considered?\n', 'answer': "\nThe presentation raises concern for pneumonia, particularly\ngiven the fever, productive cough, hypoxia, and new pulmonary\nopacity.\n\nA COPD exacerbation should also remain in the differential\ndiagnosis.\n\nThe available information should be interpreted together with\nthe patient's full clinical history, examination, imaging, and\nlaboratory findings before reaching a definitive diagnosis.\n"}


In [6]:
SYSTEM_PROMPT = """
You are a clinical AI assistant.

Provide careful, structured clinical reasoning.
Avoid making overly confident conclusions when the available
information is insufficient.
"""

def create_messages(example):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": example["question"].strip()
        },
        {
            "role": "assistant",
            "content": example["answer"].strip()
        }
    ]

    return {"messages": messages}


dataset = dataset.map(create_messages)

print(dataset[0]["messages"])

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

[{'content': '\nYou are a clinical AI assistant.\n\nProvide careful, structured clinical reasoning.\nAvoid making overly confident conclusions when the available\ninformation is insufficient.\n', 'role': 'system'}, {'content': 'A 68-year-old patient with COPD presents with fever,\nproductive cough, hypoxia, and a new pulmonary opacity.\nWhat should be considered?', 'role': 'user'}, {'content': "The presentation raises concern for pneumonia, particularly\ngiven the fever, productive cough, hypoxia, and new pulmonary\nopacity.\n\nA COPD exacerbation should also remain in the differential\ndiagnosis.\n\nThe available information should be interpreted together with\nthe patient's full clinical history, examination, imaging, and\nlaboratory findings before reaching a definitive diagnosis.", 'role': 'assistant'}]


In [7]:
dataset[0]

{'question': '\nA 68-year-old patient with COPD presents with fever,\nproductive cough, hypoxia, and a new pulmonary opacity.\nWhat should be considered?\n',
 'answer': "\nThe presentation raises concern for pneumonia, particularly\ngiven the fever, productive cough, hypoxia, and new pulmonary\nopacity.\n\nA COPD exacerbation should also remain in the differential\ndiagnosis.\n\nThe available information should be interpreted together with\nthe patient's full clinical history, examination, imaging, and\nlaboratory findings before reaching a definitive diagnosis.\n",
 'messages': [{'content': '\nYou are a clinical AI assistant.\n\nProvide careful, structured clinical reasoning.\nAvoid making overly confident conclusions when the available\ninformation is insufficient.\n',
   'role': 'system'},
  {'content': 'A 68-year-old patient with COPD presents with fever,\nproductive cough, hypoxia, and a new pulmonary opacity.\nWhat should be considered?',
   'role': 'user'},
  {'content': "The pr

In [8]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [9]:
!pip -q install -U transformers accelerate peft trl bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 46.0 MB/s eta 0:00:00


In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded!")
print("Device:", next(base_model.parameters()).device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded!
Device: cuda:0


In [11]:
test_question = """
A 72-year-old patient presents with fever, confusion,
low blood pressure, and rapid breathing.
What should be considered?
"""

In [12]:
### baseline_answer
messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": test_question.strip()
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(base_model.device)

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]

baseline_answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print(baseline_answer)

Given the symptoms of fever, confusion, low blood pressure, and rapid breathing in an elderly patient, several serious conditions need to be considered:

1. **Severe Infection**: The combination of fever, confusion, and respiratory distress could indicate sepsis or another severe infection such as pneumonia, meningitis, or endocarditis. These infections can rapidly lead to hypotension due to widespread vasodilation and release of inflammatory mediators.

2. **Cardiac Issues**: Low blood pressure (hypotension) and rapid breathing might suggest cardiac causes like heart failure, myocardial infarction, or arrhythmias that affect both the heart's pumping function and its ability to regulate blood flow throughout the body.

3. **Hypovolemic Shock**: This condition occurs when there is significant loss of blood volume leading to shock. It can result from trauma, hemorrhage, dehydration, or other factors causing rapid fluid loss.

4. **Cerebrovascular Accident (Stroke)**: Confusion and change

In [13]:
# QLoRA fine tuning .Qwen 1.5B 大概有十几亿 parameters。如果全部 fine-tune....LoRA 的思路是：原来的模型 weights 冻结，只训练额外加入的一小组参数。

In [17]:
## 释放 base model
del base_model
torch.cuda.empty_cache()

In [18]:
### 4-bit quantization， Much smaller GPU memory

from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("4-bit model loaded!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

4-bit model loaded!


In [19]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410
